# Fill a missing AdmR value: one drug

This is the first, deliberately small run of deliverable D3: fill a `NaN` in the `AdmR` column of `inputs/mode_of_administration.md` for **one drug and one ATC code**, using DailyMed label counts.

Rules applied (from the task spec):

- **Rule 3**: at most 2 routes per ATC code, intravenous excluded.
- **Rule 4**: rank routes by how many DailyMed labels report them for this ATC code.
- **Rule 5**: one output row per route.
- **Rule 7**: every label used is saved to `results/` with its DailyMed link and a timestamp; the raw API response is kept in `evidence/dailymed/`.

The source table is never edited in place; results go to a new CSV.

In [1]:
import json, re, datetime
from collections import Counter
from pathlib import Path

import pandas as pd
import requests

DRUG = "azelastine"
ATC = "S01GX07"

HERE = Path.cwd() if (Path.cwd() / "inputs").exists() else Path.cwd() / "curation"
INPUT = HERE / "inputs" / "mode_of_administration.md"
VOCAB = HERE / "inputs" / "route_vocabulary.csv"
CACHE = HERE / "evidence" / "dailymed" / f"{DRUG}_spls.json"
RESULTS = HERE / "results"
RESULTS.mkdir(exist_ok=True)

## 1. Load the table and find the row

In [2]:
rows = [l.strip().strip("|").split(" | ") for l in INPUT.read_text().splitlines() if l.startswith("|")]
rows = [[c.strip() for c in r] for r in rows if len(r) >= 4]
admr = pd.DataFrame(rows[2:], columns=rows[0])
print(len(admr), "rows,", (admr.AdmR == "NaN").sum(), "with AdmR = NaN")
admr[admr.GenericName == DRUG]

1167 rows, 315 with AdmR = NaN


,CID,GenericName,ATC,AdmR
11,2267,azelastine,R06AX19,O
12,2267,azelastine,S01GX07,NaN
13,2267,azelastine,R01AC03,N


## 1b. Route vocabulary

The WHO codes in the table are coarse: `P` covers every injection, and there is no code for eye, ear, skin or intrathecal use. `inputs/route_vocabulary.csv` replaces them with 27 specific routes taken from the *Routes of Drug Administration* article (Pharmacology Mentor), in its four groups:

- **enteral**: oral, sublingual, buccal, rectal
- **parenteral**: intravenous, intramuscular, subcutaneous, intradermal
- **other**: transdermal, inhalational, topical, intranasal, ocular, otic, vaginal
- **specialised** (Table 1): intrathecal, epidural, intraosseous, intra-articular, intra-arterial, intraperitoneal, intravesical, intraventricular, intracameral

It also includes urethral and implant, because the table already uses them. Each route keeps a `who_ddd_code` column, so a fine route can always be rolled back up to the WHO code (intrathecal → `P`). Two regular expressions recognise a route in a DailyMed label title: `label_pattern` holds words that name the route (INTRATHECAL, BUCCAL, OTIC) and `form_pattern` holds dosage forms that only imply it (TABLET, SPRAY, INJECTION).

In [3]:
vocab = pd.read_csv(VOCAB, dtype=str).fillna("")
vocab

,route_code,route,group,who_ddd_code,label_pattern,form_pattern,source
0,O,oral,enteral,O,\bORAL,TABLET|CAPSULE|SYRUP|ELIXIR|GRANULE,Pharmacology Mentor: Oral Route
1,SL,sublingual,enteral,SL,SUBLINGUAL,,Pharmacology Mentor: Sublingual Route
2,BUC,buccal,enteral,,BUCCAL,,Pharmacology Mentor: Buccal Route
3,R,rectal,enteral,R,RECTAL,SUPPOSITORY|ENEMA,Pharmacology Mentor: Rectal Route
4,IV,intravenous,parenteral,P,INTRAVENOUS,,Pharmacology Mentor: Intravenous Route
5,IM,intramuscular,parenteral,P,INTRAMUSCULAR,,Pharmacology Mentor: Intramuscular Route
6,SC,subcutaneous,parenteral,P,SUBCUTANEOUS,,Pharmacology Mentor: Subcutaneous Route
7,ID,intradermal,parenteral,P,INTRADERMAL,,Pharmacology Mentor: Intradermal Route
8,P,"parenteral, route not specified",parenteral,P,PARENTERAL,INJECTION|INJECTABLE,WHO DDD code P (coarse; refine when the label ...
9,TD,transdermal,other,TD,TRANSDERMAL,PATCH,Pharmacology Mentor: Transdermal Route


How the codes already in the table map onto the vocabulary. `P` cannot be refined from the table alone; it needs the label route for each drug, so those rows are flagged for review rather than guessed.

In [4]:
WHO_TO_ROUTE = {"O": "O", "SL": "SL", "R": "R", "TD": "TD", "TD patch": "TD", "N": "N", "V": "V",
                "Inhal": "INH", "Inhal.powder": "INH", "Inhal.solution": "INH", "Inhal.aerosol": "INH",
                "oral aerosol": "O", "Chewing gum": "O", "urethral": "URE", "implant": "IMPL",
                "s.c. implant": "SC", "ointment": "TOP", "lamella": "OPH", "P": "P"}
mapped = admr.AdmR.map(WHO_TO_ROUTE)
print("unmapped existing codes:", sorted(set(admr.AdmR[mapped.isna() & (admr.AdmR != "NaN")])))
admr.assign(route_code=mapped).groupby(["AdmR", "route_code"], dropna=False).size().sort_values(ascending=False).head(15)

unmapped existing codes: []


AdmR            route_code
O               O             477
NaN             NaN           315
P               P             237
R               R              26
N               N              22
Inhal.powder    INH            19
Inhal.aerosol   INH            13
Inhal.solution  INH            13
TD              TD             11
SL              SL             10
V               V              10
implant         IMPL            4
Inhal           INH             3
Chewing gum     O               1
TD patch        TD              1
dtype: int64

The existing `AdmR` codes are WHO ATC/DDD route codes (O, P, N, R, TD, SL, ...). WHO assigns no DDD, and so no route, to most eye (S01), anaesthetic (N01) and skin (D) products, which is why so many of these rows are empty.

In [5]:
admr.AdmR.value_counts().head(12)

AdmR
O                 477
NaN               315
P                 237
R                  26
N                  22
Inhal.powder       19
Inhal.solution     13
Inhal.aerosol      13
TD                 11
V                  10
SL                 10
implant             4
Name: count, dtype: int64

## 2. Get the DailyMed labels

Live call to the DailyMed SPL API. If the network blocks it, the notebook falls back to the saved response in `evidence/dailymed/`, so the run is reproducible offline.

In [6]:
def dailymed_spls(name):
    url = "https://dailymed.nlm.nih.gov/dailymed/services/v2/spls.json"
    out, page = [], 1
    while True:
        r = requests.get(url, params={"drug_name": name, "pagesize": 100, "page": page}, timeout=20)
        r.raise_for_status()
        j = r.json()
        out += [{"setid": d["setid"], "title": d["title"]} for d in j["data"]]
        if j["metadata"]["next_page"] in (None, "null"):
            return out

try:
    labels = dailymed_spls(DRUG)
    retrieved = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
    CACHE.write_text(json.dumps({"retrieved_utc": retrieved, "labels": labels}, indent=1))
    source = "live"
except Exception as e:
    cached = json.loads(CACHE.read_text())
    labels, retrieved, source = cached["labels"], cached["retrieved_utc"], f"cache ({type(e).__name__})"
print(len(labels), "labels from", source, "retrieved", retrieved)

47 labels from cache (ProxyError) retrieved 2026-10-09T03:37:06+00:00


## 3. Route of each label, and which ATC code it belongs to

Route words are checked before dosage forms, so "INTRATHECAL INJECTION" is intrathecal (not generic `P`), "BUCCAL TABLET" is buccal (not oral) and "OTIC SOLUTION/DROPS" is otic (not ocular). The table at the end of this section checks that on example titles. A drug can have several ATC codes (azelastine: R06AX19 oral, R01AC03 nasal, S01GX07 eye), so only labels whose route fits **this** ATC code are counted (rule 4 is per ATC).

In [7]:
ORDER = {"specialised": 0, "parenteral": 1, "enteral": 2, "other": 3}
ranked = vocab.assign(rank=vocab.group.map(ORDER)).sort_values("rank", kind="stable")
# pass 1: words that name the route; pass 2: dosage forms that only imply it
matchers = [(r.route_code, r.label_pattern) for r in ranked.itertuples() if r.label_pattern] + \
           [(r.route_code, r.form_pattern) for r in ranked.itertuples() if r.form_pattern]
ATC_PREFIX_ROUTES = {"S01": {"OPH", "ICAM"}, "S02": {"OT"}, "R01": {"N"}, "R06": {"O"}}

def route_of(title):
    for code, pattern in matchers:
        if re.search(pattern, title.upper()):
            return code
    return "unknown"

checks = {"LIORESAL INTRATHECAL (BACLOFEN) INJECTION": "IT", "MORPHINE SULFATE INJECTION": "P",
          "CIPROFLOXACIN OTIC SOLUTION/ DROPS": "OT", "FENTANYL BUCCAL TABLET": "BUC",
          "TRIAMCINOLONE INJECTION, SUSPENSION INTRA-ARTICULAR": "IAR", "BUPIVACAINE EPIDURAL INJECTION": "EPI",
          "NICOTINE TRANSDERMAL SYSTEM PATCH": "TD", "AZELASTINE HYDROCHLORIDE SPRAY, METERED": "N"}
assert all(route_of(t) == want for t, want in checks.items()), {t: route_of(t) for t in checks}

lab = pd.DataFrame(labels)
lab["route_code"] = lab.title.map(route_of)
allowed = ATC_PREFIX_ROUTES.get(ATC[:3])
lab["counts_for_this_ATC"] = lab.route_code.isin(allowed) if allowed else True
lab.route_code.value_counts()

route_code
N      38
OPH     9
Name: count, dtype: int64

## 4. Apply rules 3 and 4: drop IV, keep the top 2

In [8]:
counts = Counter(lab.loc[lab.counts_for_this_ATC & (lab.route_code != "IV"), "route_code"])
top = counts.most_common(2)
print(f"{ATC}: route counts {dict(counts)} -> keep {top}")

S01GX07: route counts {'OPH': 9} -> keep [('OPH', 9)]


## 5. Rule 5: one row per route, then save

In [9]:
n_total = len(lab)
v = vocab.set_index("route_code")
out = pd.DataFrame([{
    "CID": admr.loc[(admr.GenericName == DRUG) & (admr.ATC == ATC), "CID"].iloc[0],
    "GenericName": DRUG, "ATC": ATC, "AdmR_before": "NaN",
    "AdmR_after": v.at[code, "route"], "route_code": code, "route_group": v.at[code, "group"],
    "who_ddd_code": v.at[code, "who_ddd_code"],
    "Edit_Flag": "filled_from_DailyMed",
    "evidence": f"{n} of {n_total} DailyMed labels for {DRUG} have this route",
} for code, n in top])
out.to_csv(RESULTS / f"admr_{DRUG}_{ATC}.csv", index=False)

prov = lab.assign(drug=DRUG, source_url="https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid=" + lab.setid,
                  retrieved_utc=retrieved)
prov.to_csv(RESULTS / f"provenance_{DRUG}.csv", index=False)
out

,CID,GenericName,ATC,AdmR_before,AdmR_after,route_code,route_group,who_ddd_code,Edit_Flag,evidence
0,2267,azelastine,S01GX07,NaN,ocular (ophthalmic),OPH,other,,filled_from_DailyMed,9 of 47 DailyMed labels for azelastine have th...


## What is still open

- IV-only drugs such as alfentanil (N01AH02) have no route left after rule 3; how to record them is waiting on Star.
- The 237 existing `P` rows could be refined (IV, IM, SC, intrathecal, epidural...) from each drug's labels; that is a separate run.
- `who_ddd_code` is blank where WHO has no code (buccal, topical, ocular, otic, intracameral).
- PubChem, WHO ATC/DDD and DrugBank are blocked from the cloud session that ran this notebook, so only DailyMed was used.